In [4]:
import numpy as np
import pandas as pd

def run_module_4_simulation(df_plan, scenario="Cơ sở", n_simulations=100):
    """
    MODULE 4: MÔ PHỎNG CHÍNH SÁCH VÀ ĐÁNH GIÁ TÁC ĐỘNG (STRESS TESTING)
    Nhận kế hoạch phân bổ THỰC TẾ từ Module 3 và kiểm định sức chống chịu dưới các cú sốc vĩ mô.
    """
    np.random.seed(42)
    
    # 1. KHỞI TẠO CÁC BIẾN TRẠNG THÁI NỀN KINH TẾ (Năm gốc 2026)
    K_init = 27500.0   # Vốn vật chất (Tỷ VND)
    D_init = 20.3      # Chỉ số số hóa (% GDP)
    AI_init = 86.0     # Doanh nghiệp AI (Nghìn DN)
    H_init = 30.0      # Vốn nhân lực (%)
    L = 54.0           # Lực lượng lao động (Triệu người)
    
    years = sorted(df_plan["Năm"].unique())
    records = []
    
    # 2. ĐỊNH NGHĨA THAM SỐ CÚ SỐC THEO KỊCH BẢN
    shock_gdp = 0.0
    shock_cyber = 0.0
    shock_unemploy = 0.0
    
    if scenario == "Khủng hoảng An ninh mạng":
        shock_gdp = -0.05    
        shock_cyber = 0.40   
    elif scenario == "Đột phá AI":
        shock_gdp = 0.08     
        shock_unemploy = 0.35 

    # 3. VÒNG LẶP MÔ PHỎNG QUA CÁC NĂM
    K, D, AI, H = K_init, D_init, AI_init, H_init
    
    for year in years:
        df_year = df_plan[df_plan["Năm"] == year]
        
        alloc_I = df_year[df_year["Hạng mục"].str.contains("Hạ tầng")]["Ngân sách (Tỷ VND)"].sum()
        alloc_D = df_year[df_year["Hạng mục"].str.contains("Chuyển đổi")]["Ngân sách (Tỷ VND)"].sum()
        alloc_AI = df_year[df_year["Hạng mục"].str.contains("Trí tuệ")]["Ngân sách (Tỷ VND)"].sum()
        alloc_H = df_year[df_year["Hạng mục"].str.contains("Nhân lực")]["Ngân sách (Tỷ VND)"].sum()
        
        # Tạo tiếng ồn thị trường ngẫu nhiên (Monte Carlo)
        market_noise = np.random.normal(0, 0.02) 
        
        K += alloc_I * 1.0
        D += (alloc_D / 100) * (1.0 + shock_gdp)
        AI += (alloc_AI / 20) * (1.0 + (0.2 if scenario == "Đột phá AI" else 0.0))
        H += alloc_H / 200
        
        # Hàm sản xuất Cobb-Douglas mở rộng
        A_tfp = 35.0 * (1.0 + shock_gdp + market_noise)
        current_GDP = A_tfp * (K**0.33) * (L**0.42) * (D**0.10) * (AI**0.08) * (H**0.07)
        
        # Tính toán các chỉ số rủi ro / vệ tinh
        unemploy_risk = max(0, 0.6 * (alloc_AI/1000) - 0.25 * (H/100) + shock_unemploy)
        cyber_risk = max(0, 0.4 * (alloc_D/1000) + 0.5 * (alloc_AI/1000) - 0.3 * (H/100) + shock_cyber)
        emission = max(0, 0.5 * (alloc_I/1000) + 0.3 * (alloc_AI/1000) - 0.2 * (alloc_D/1000))
        
        records.append({
            "Kịch bản": scenario,
            "Năm": year,
            "GDP Thực tế (Tỷ VND)": current_GDP,
            "Rủi ro Thất nghiệp": unemploy_risk,
            "Rủi ro Mạng": cyber_risk,
            "Mức phát thải CO2": emission,
            "Vốn vật chất K": K,
            "Hạ tầng số D": D
        })
        
    return pd.DataFrame(records)

# =====================================================================
# CHẠY TÍCH HỢP ĐÚNG KẾT QUẢ ĐẦU RA THỰC TẾ MODULE 3 CỦA BẠN
# =====================================================================
if __name__ == "__main__":
    print("Đang nạp kết quả tối ưu từ Module 3 của bạn...")
    
    # Khai báo chính xác bảng dữ liệu đầu ra dựa trên kết quả tối ưu của bạn
    actual_m3_records = [
        # --- Năm 2026 (Tổng ngân sách: 50,000 tỷ) ---
        {"Năm": 2026, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 15000.00},
        {"Năm": 2026, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 2500.00},
        {"Năm": 2026, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 5000.00},
        {"Năm": 2026, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 15000.00},
        {"Năm": 2026, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 5000.00},
        {"Năm": 2026, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 2500.00},
        {"Năm": 2026, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 5000.00},
        
        # --- Năm 2027 (Tổng ngân sách: 55,000 tỷ) ---
        {"Năm": 2027, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 16500.00},
        {"Năm": 2027, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 2750.00},
        {"Năm": 2027, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 5500.00},
        {"Năm": 2027, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 16500.00},
        {"Năm": 2027, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 5500.00},
        {"Năm": 2027, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 2750.00},
        {"Năm": 2027, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 5500.00},

        # --- Năm 2028 (Tổng ngân sách: 60,000 tỷ) ---
        {"Năm": 2028, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 18000.00},
        {"Năm": 2028, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3000.00},
        {"Năm": 2028, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 6000.00},
        {"Năm": 2028, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 18000.00},
        {"Năm": 2028, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 6000.00},
        {"Năm": 2028, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3000.00},
        {"Năm": 2028, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 6000.00},

        # --- Năm 2029 (Tổng ngân sách: 65,000 tỷ) ---
        {"Năm": 2029, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 19500.00},
        {"Năm": 2029, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3250.00},
        {"Năm": 2029, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 6500.00},
        {"Năm": 2029, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 19500.00},
        {"Năm": 2029, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 6500.00},
        {"Năm": 2029, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3250.00},
        {"Năm": 2029, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 6500.00},

        # --- Năm 2030 (Tổng ngân sách: 70,000 tỷ) ---
        {"Năm": 2030, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 21000.00},
        {"Năm": 2030, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3500.00},
        {"Năm": 2030, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 7000.00},
        {"Năm": 2030, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 21000.00},
        {"Năm": 2030, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 7000.00},
        {"Năm": 2030, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3500.00},
        {"Năm": 2030, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 7000.00},
    ]
    df_m3_plan = pd.DataFrame(actual_m3_records)

    # THỰC THI MÔ PHỎNG ĐA KỊCH BẢN (STRESS TESTING)
    print("Đang tiến hành chạy Module 4 trên nền phương án phân bổ của bạn...")
    df_base = run_module_4_simulation(df_m3_plan, scenario="Cơ sở")
    df_cyber = run_module_4_simulation(df_m3_plan, scenario="Khủng hoảng An ninh mạng")
    df_ai_break = run_module_4_simulation(df_m3_plan, scenario="Đột phá AI")
    
    # Gộp kết quả của cả 3 kịch bản chính sách
    df_m4_final = pd.concat([df_base, df_cyber, df_ai_break], ignore_index=True)
    
    # IN BÁO CÁO KẾT QUẢ ĐÁNH GIÁ TÁC ĐỘNG CUỐI KỲ (NĂM CUỐI 2030)
    print("\n" + "="*95)
    print("KẾT QUẢ ĐÁNH GIÁ TÁC ĐỘNG VĨ MÔ THỰC TẾ (MODULE 4) - TRẠNG THÁI NĂM CUỐI 2030:")
    print("="*95)
    
    df_2030 = df_m4_final[df_m4_final["Năm"] == 2030][["Kịch bản", "GDP Thực tế (Tỷ VND)", "Rủi ro Thất nghiệp", "Rủi ro Mạng"]]
    pd.set_option('display.float_format', lambda x: '%.2f' % x)
    print(df_2030.to_string(index=False))
    print("="*95)

Đang nạp kết quả tối ưu từ Module 3 của bạn...
Đang tiến hành chạy Module 4 trên nền phương án phân bổ của bạn...

KẾT QUẢ ĐÁNH GIÁ TÁC ĐỘNG VĨ MÔ THỰC TẾ (MODULE 4) - TRẠNG THÁI NĂM CUỐI 2030:
                Kịch bản  GDP Thực tế (Tỷ VND)  Rủi ro Thất nghiệp  Rủi ro Mạng
                   Cơ sở              49328.15                3.56        13.94
Khủng hoảng An ninh mạng              46614.52                3.56        14.34
              Đột phá AI              54447.48                3.91        13.94
